# Week 47, Day 2: Spikes and fields: measuring phase locking without bias

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w047/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Phases from a filtered field

Band-pass the field between 6 and 10 Hz (third-order Butterworth, filtfilt), take the phase of its Hilbert transform, and read it at the spike times. Write plv(phases) and ppc(phases) using the second form of the PPC formula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert
r = np.random.default_rng(2010)
fs, T = 1000, 60
t = np.arange(T * fs) / fs
theta = 2 * np.pi * 8 * t + np.cumsum(0.05 * r.normal(size=len(t)))      # a slightly wandering rhythm
lfp = np.cos(theta) + 0.8 * r.normal(size=len(t))

def spikes(rate, kappa, pref=0.0):
    lam = rate * np.exp(kappa * np.cos(theta - pref)) / np.i0(kappa) / fs
    return np.where(r.random(len(t)) < lam)[0]

b, a = butter(3, [6 / (fs / 2), 10 / (fs / 2)], "band")
phase = ...

def plv(ph):
    return ...

def ppc(ph):
    n = len(ph)
    return ...

s = spikes(5, 1.0); p = phase[s]
print(f"{len(s)} spikes: PLV {plv(p):.3f}, PPC {ppc(p):.3f}, PLV squared {plv(p) ** 2:.3f}")
plt.hist(p, 24); plt.xlabel("theta phase at spike (rad)"); plt.show()

In [ ]:
import numpy as np
def test_locked():
    "the PLV of the locked neuron is between 0.3 and 0.5"
    assert 0.3 < plv(p) < 0.5
def test_ppc():
    "its PPC is close to the PLV squared"
    assert abs(ppc(p) - plv(p) ** 2) < 0.02

test_locked()
test_ppc()
print("All checks passed")

### Exercise: Unlocked spikes, few and many

For n in 10, 30, 100 and 1000, draw 200 sets of n uniformly random phases and average the PLV and PPC over sets.

In [ ]:
mean_plv, mean_ppc = {}, {}
for n in [10, 30, 100, 1000]:
    sets = [r.uniform(-np.pi, np.pi, n) for _ in range(200)]
    mean_plv[n] = ...
    mean_ppc[n] = ...
    print(f"n = {n:4d}: mean PLV {mean_plv[n]:.3f} (1/sqrt(n) = {1 / np.sqrt(n):.3f}), mean PPC {mean_ppc[n]:+.4f}")

In [ ]:
import numpy as np
def test_plv_bias():
    "for unlocked spikes, the PLV with 10 spikes exceeds 0.2 and falls with n"
    assert mean_plv[10] > 0.2 and mean_plv[10] > mean_plv[100] > mean_plv[1000]
def test_ppc_unbiased():
    "the PPC averages within 0.02 of zero at every n"
    assert all(abs(mean_ppc[n]) < 0.02 for n in mean_ppc)

test_plv_bias()
test_ppc_unbiased()
print("All checks passed")

### Exercise: Same locking, different rates

Using the first 20 seconds of the field, simulate both neurons 200 times each with kappa 0.3 and average their PLV and PPC.

In [ ]:
t20 = 20 * fs
res = {}
for name, rate in [("0.5 Hz", 0.5), ("20 Hz", 20.0)]:
    vals = []
    for _ in range(200):
        s = spikes(rate, 0.3); s = s[s < t20]
        ...
    res[name] = np.mean(vals, 0)
    print(f"{name:5s}: PLV {res[name][0]:.3f}, PPC {res[name][1]:.3f}")

In [ ]:
import numpy as np
def test_plv_misleads():
    "the PLV makes the low-rate neuron look more locked"
    assert res["0.5 Hz"][0] > res["20 Hz"][0] + 0.1
def test_ppc_fair():
    "the PPCs agree within 0.04"
    assert abs(res["0.5 Hz"][1] - res["20 Hz"][1]) < 0.02

test_plv_misleads()
test_ppc_fair()
print("All checks passed")